# Análisis Exploratorio de Datos - Primer acercamiento

En este notebook vamos a tratar de entender los principales patrones que siguen los datos con los que vamos a trabajar. De primeras sabemos que el dataset es un histórico de reserva de hoteles y que el problema consistirá en clasificar si la reserva será cancelada o no.

## Hipótesis iniciales
Para que nuestro análisis siga una dirección concreta, vamos a plantear una serie de hipótesis sobre el comportamiento de los clientes, las características de la reserva y las condiciones del hotel.

**Antelación de la reserva**
- Cuanto mayor sea la antelación con la que se hizo la reserva, mayor será la probabilidad de cancelación.
- En cambio las de menor cancelación tienen menor probabilidad, ya que podrían responder a necesidades de alojamiento inmediatas.
- La relación entre antelación y tasa de cancelación no es lineal: podría existir un umbral de antelación que dispare la probabilidad de cancelación.

**Características del cliente**
- Los clientes que ya realizaron reservas en otras ocasiones son menos propensos a cancelar.
- Los clientes que ya cancelaron en otras ocasiones son más probables a volver a cancelar.

**Características económicas**
- Las reservas de mayor coste tienen mayor probabilidad de cancelación.
- Las reservas que requieren un depósito no reembolsable presentan una menor tasa de cancelación. En cambio, las de depósito reembolsable o sin depósito presentan una mayor probabilidad de cancelación.

**Canal de distribución**
- Las reservas realizadas mediante agencias de viajes (TA/TO) presentan una tasa de cancelación diferente a las realizadas directamente con el hotel.
- Las reservas online tienen mayor probabilidad de cancelación que las realizadas por otros canales.
- Los clientes que vienen por parte de empresas tienen menor probabilidad de cancelar.

**Características temporales**
- Las reservas correspondientes a temporadas de alta demanda presentan una menor tasa de cancelación.
- La tasa de cancelación varía dependiendo del mes de llegada.
- Las reservas realizadas para fines de semana presentan un comportamiento de cancelación diferente a las reservas entre semana.

**Características de la estancia**
- Las reservas con mayor duración tienen una mayor probabilidad de cancelación.
- Las reservas realizadas para grupos o familias presentan un comportamiento de cancelación diferente al de las reservas individuales.
- Las reservas que incluyen solicitudes especiales presentan una menor probabilidad de cancelación, debido a que son clientes más comprometidos con la estancia.

In [1]:
# Importamos librerías necesarias
import pandas as pd

## Preprocesamiento de datos

In [ ]:
from pathlib import Path

ROOT_PATH = Path("../..")
DATA_PATH = ROOT_PATH / "data" / "raw" / "dataset_practica_final.csv"

# Cargamos el dataset
reservas_df = pd.read_csv(DATA_PATH)

reservas_df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [ ]:
# Imprimimos información general del DataFrame
print("___"*10, "INFORMACIÓN", "___"*10)
print(reservas_df.info())
print("==="*24)



______________________________ INFORMACIÓN ______________________________
<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies         

Vamos a empezar por agrupar las variables según su naturaleza (categóricas, numéricas, booleanas).

### Categóricas nominales

Estas variables representan grupos o etiquetas sin jerarquía.
Variable	Descripción
hotel	Tipo de hotel
arrival_date_month	Mes de llegada
meal	Tipo de régimen alimenticio
country	País de procedencia
market_segment	Segmento de mercado
distribution_channel	Canal de distribución
reserved_room_type	Tipo de habitación reservada
assigned_room_type	Tipo de habitación asignada
deposit_type	Tipo de depósito
agent	Identificador de la agencia
company	Identificador de la empresa
reservation_status	Estado final de la reserva
reservation_status_date	Fecha del último cambio de estado

**Observaciones**
- agent y company son identificadores.
- arrival_date_month tiene una naturaleza temporal (meses del año), pero la vamos a considerar nominal sin jerarquía por simplicidad.
- reservation_status_date debería convertirse a datetime para trabajar con ella como variable temporal. De aquí podrían salir nuevas características.



### Categóricas ordinales (jerárquicas)
Podrían tratarse como nominales también, pero podemos asignarle la siguiente jerarquía.

Variable	Posible orden	Observaciones
customer_type	Contract, Group, Transient, Transient-Party	No existe un orden jerárquico universal.
deposit_type	No Deposit, Refundable, Non Refund	El orden depende de cómo definamos la dimensión de comparación.

### Numéricas discretas
